# Amazon ML Challenge - Kaggle Runner (checkpointed)
One notebook, start to finish: install -> locate data -> train -> predict -> zip outputs.
Every stage checkpoints to `/kaggle/working/output/checkpoints/`, so if the session dies you just re-run the same cell with `--resume` and it picks up where it stopped.

**If you change cleaning/blocking/feature code between runs, delete the checkpoints folder first** (cell provided below) - stale checkpoints are not auto-invalidated.

## 1. Config - the ONLY thing you edit

In [ ]:
import os

# Point this at the Kaggle dataset you added via 'Add Data' (the folder that
# contains student_resource/ or dataset/). Check the right-hand Data panel.
DATA_ROOT = "/kaggle/input/amazon-ml-challenge"   # <-- EDIT THIS

REPO_URL = "https://github.com/0xmaster7/amazon-ml-challenge"
REPO_DIR = "/kaggle/working/repo"
OUTPUT_DIR = "/kaggle/working/output"

def find_split(root, split):
    for cand in (os.path.join(root, 'student_resource', 'dataset', split),
                 os.path.join(root, 'dataset', split),
                 os.path.join(root, split)):
        if os.path.isdir(cand):
            return cand
    raise FileNotFoundError(f'Could not find {split}/ under {root} - fix DATA_ROOT')

TRAIN_DIR = find_split(DATA_ROOT, 'train')
TEST_DIR = find_split(DATA_ROOT, 'test')
print('train:', TRAIN_DIR, '->', sorted(os.listdir(TRAIN_DIR)))
print('test :', TEST_DIR, '->', sorted(os.listdir(TEST_DIR)))

## 2. Install deps + get the code

In [ ]:
!pip install -q rapidfuzz scikit-learn xgboost sentence-transformers faiss-cpu transformers accelerate bitsandbytes
!git clone $REPO_URL $REPO_DIR || (cd $REPO_DIR && git pull)

# If your fixed files are NOT pushed to GitHub yet: upload amlc_fixpack.zip as a
# Kaggle dataset, then uncomment and run this line instead:
# !unzip -o /kaggle/input/amlc-fixpack/amlc_fixpack.zip -d $REPO_DIR

SRC = os.path.join(REPO_DIR, 'code', 'business_entity_resolution', 'src')
print(sorted(os.listdir(SRC)))

## 3. (Optional) Fresh start - deletes checkpoints and cached embeddings

In [ ]:
# Run this cell ONLY when you changed cleaning/blocking/feature code
# and need a clean rebuild.
# !rm -rf /kaggle/working/output/checkpoints
# !rm -f /kaggle/working/pool_embeddings.dat* /kaggle/working/s1_embeddings.dat*

## 4. Train (blocking + features + XGBoost + threshold calibration)
Flags: `--resume` picks up from the last checkpoint. `--seeds 42 1337 2024` ensembles seeds per fold (3x slower, usually +F). `--stratify` tunes per-country OOF thresholds. `--embedder2 <model>` adds a second embedder for recall coverage. `--no_spw` turns off scale_pos_weight (ablation; default ON is better under F_0.5).

In [ ]:
!python $SRC/train.py \
    --data_dir $TRAIN_DIR \
    --output_dir $OUTPUT_DIR \
    --resume --skip_llm

# When the score looks good, do one final run WITHOUT --skip_llm to let the
# LLM arbitrage borderline pairs AND measure its validation delta.

## 5. Predict on test -> matching_results.tsv + candidate_pairs.tsv
Per-country thresholds applied automatically from the saved model. `--no_conflict_resolution` disables the one-S1-per-pool-ID post-pass (only if train.py's GT check warned). `--band_lo/--band_hi` widen/narrow the LLM borderline band (see error_analysis.py output for where errors cluster).

In [ ]:
!python $SRC/predict.py \
    --test_dir $TEST_DIR \
    --model_path $OUTPUT_DIR/xgb_model.pkl \
    --output_dir $OUTPUT_DIR \
    --resume --skip_llm

## 6. Package outputs for submission

In [ ]:
!cd $OUTPUT_DIR && zip -j /kaggle/working/submission_outputs.zip matching_results.tsv candidate_pairs.tsv
print('Download submission_outputs.zip from the Output panel, or upload the two TSVs directly.')

## 7. Pre-submit checklist (run these before every submission)
1. Smoke tests: scorer edge cases + synthetic French record end-to-end.
2. Official `utils/validate_submission.py` on both output files.
3. Optional: `error_analysis.py` to see where OOF errors cluster (needs train checkpoints).
4. Optional one-time: `finetune_embedder.py` for the biggest single recall gain - then delete checkpoints and re-run blocking+training.

In [ ]:
!python $SRC/smoke_tests.py

# Official validator (real args: --matching, --candidate, --test-dir)
# Path is wherever the competition starter kit lives in your dataset.
import os
VAL = os.path.join(os.path.dirname(DATASET), 'utils', 'validate_submission.py') if 'DATASET' in dir() else None
if VAL and os.path.exists(VAL):
    !python $VAL --matching $OUTPUT_DIR/matching_results.tsv --candidate $OUTPUT_DIR/candidate_pairs.tsv --test-dir $TEST_DIR
else:
    print('Validator not found at', VAL, '- predict.py already self-validated.')

# Optional: where are my OOF errors?
# !python $SRC/error_analysis.py --data_dir $TRAIN_DIR --output_dir $OUTPUT_DIR